In [ ]:
import re
import os
import pandas as pd

def parse_markdown_table(table_str, manual_header=None):
    """
    Converte una stringa contenente una tabella in formato markdown in un DataFrame pandas.
    Se viene individuata una riga separatrice (la seconda riga di trattini),
    la prima riga viene usata come header.
    Altrimenti, se viene fornito manual_header, verrà usato quello.
    """
    rows = table_str.splitlines()
    rows = [row.strip() for row in rows if row.strip()]
    
    header = None
    data_rows = []
    
    if len(rows) >= 2:
        first_row_cells = [cell.strip() for cell in rows[0].split("|") if cell.strip()]
        second_row = rows[1]
        if re.match(r'^[\-\|\s]+$', second_row):
            header = first_row_cells
            data_rows = rows[2:]
        else:
            header = manual_header if manual_header is not None else first_row_cells
            data_rows = rows[1:]
    else:
        cells = [cell.strip() for cell in rows[0].split("|") if cell.strip()]
        header = manual_header if manual_header is not None else [f"Column{i+1}" for i in range(len(cells))]
        data_rows = rows

    data = []
    for row in data_rows:
        cells = [cell.strip() for cell in row.split("|") if cell.strip()]
        if len(cells) == len(header):
            data.append(cells)
    df = pd.DataFrame(data, columns=header)
    return df

def color_success(val):
    """
    Restituisce una stringa di stile CSS:
      - "color: green" se il testo (case-insensitive) contiene "yes"
      - "color: red" se contiene "no"
    """
    if isinstance(val, str):
        lower_val = val.lower()
        if "yes" in lower_val:
            return "color: green"
        elif "no" in lower_val:
            return "color: red"
    return ""

def parse_md_blocks(md_text):
    """
    Divide il testo markdown in blocchi (test blocks).
    Ogni blocco è un dizionario con chiavi:
      - 'title': il titolo principale (ad esempio, "3GPPTR38_901_UMa_C1_uniform results")
      - 'run_time': la linea contenente il run time (se presente)
      - 'mean_table': le linee della tabella "mean" (se presenti)
      - 'median_table': le linee della tabella "median" (se presenti)
    """
    lines = md_text.splitlines()
    blocks = []
    current_block = None
    current_subsection = None

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("##"):
            header_text = stripped.lstrip("#").strip()
            if header_text.lower() in ["mean", "median"]:
                current_subsection = header_text.lower()
                if current_block is not None:
                    current_block[current_subsection + "_table"] = []
            else:
                if current_block is not None:
                    blocks.append(current_block)
                current_block = {"title": header_text, "run_time": None}
                current_subsection = None
        else:
            if "**Run time**" in stripped:
                m = re.search(r"\*\*Run time\*\*:\s*(.+)", stripped)
                if m and current_block is not None:
                    current_block["run_time"] = m.group(1).strip()
            elif stripped.startswith("|") and current_subsection in ["mean", "median"]:
                current_block[current_subsection + "_table"].append(stripped)
    if current_block is not None:
        blocks.append(current_block)
    return blocks

def main():
    # Legge il file markdown dalla directory ../outputs/
    input_path = os.path.join("..", "outputs", "Summary.md")
    output_path = os.path.join("..", "outputs", "Summary.html")
    
    with open(input_path, "r", encoding="utf-8") as f:
        md_text = f.read()
    
    blocks = parse_md_blocks(md_text)
    manual_header = ["Attribute", "Expected", "Actual", "Difference", "Success"]
    
    html_sections = []
    for block in blocks:
        section_html = f"<h2>{block.get('title', '')}</h2>\n"
        if block.get("run_time"):
            section_html += f"<p><strong>Run time:</strong> {block['run_time']}</p>\n"
        
        if "mean_table" in block:
            mean_table_str = "\n".join(block["mean_table"])
            try:
                df_mean = parse_markdown_table(mean_table_str, manual_header=manual_header)
                if "Success" in df_mean.columns:
                    styled_mean = df_mean.style.map(color_success, subset=["Success"])
                    mean_html = styled_mean.to_html()
                else:
                    mean_html = df_mean.to_html()
                section_html += f"<h3>Mean</h3>\n" + mean_html + "\n"
            except Exception as e:
                section_html += f"<p>Error parsing mean table: {e}</p>\n"
        
        if "median_table" in block:
            median_table_str = "\n".join(block["median_table"])
            try:
                df_median = parse_markdown_table(median_table_str, manual_header=manual_header)
                if "Success" in df_median.columns:
                    styled_median = df_median.style.map(color_success, subset=["Success"])
                    median_html = styled_median.to_html()
                else:
                    median_html = df_median.to_html()
                section_html += f"<h3>Median</h3>\n" + median_html + "\n"
            except Exception as e:
                section_html += f"<p>Error parsing median table: {e}</p>\n"
        
        html_sections.append(section_html)
    
    full_html = "<html><head><meta charset='utf-8'><title>Summary</title></head><body>"
    for section in html_sections:
        full_html += section + "<br><hr><br>"
    full_html += "</body></html>"
    
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(full_html)
    
    print("Generated", output_path, "with", len(html_sections), "test blocks.")

if __name__ == "__main__":
    main()

    from IPython.display import HTML, display
    output_path = os.path.join("..", "outputs", "Summary.html")
    with open(output_path, "r", encoding="utf-8") as f:
        html_content = f.read()
    display(HTML(html_content))
    
    # Rimuove il file HTML generato
    os.remove(output_path)
    print(f"{output_path} has been removed.")
